In [10]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
from pcm_pix.io import load_materials

m = load_materials(CFG, base_dir="data")

sb2se3_am = m.sb2se3_am
sb2se3_cr = m.sb2se3_cr
gst_am = m.gst_am
gst_cr = m.gst_cr

logger.info("materials loaded: sb2se3_am=%s, sb2se3_cr=%s", len(sb2se3_am), len(sb2se3_cr))

2026-03-03 12:17:19,917 | INFO | materials loaded: sb2se3_am=171, sb2se3_cr=171


In [12]:
CFG["mesh_names"] = [
    "Sb2Se3 - amorphous_mesh_sbse_2025.txt",
    "Sb2Se3 - crystal_mesh12_sbse_2025.txt",
]

In [14]:
import importlib
import pcm_pix.features as f
importlib.reload(f)

<module 'pcm_pix.features' from '/home/slava/КВАНТЭЛ/GIT/PCM_pix/pcm_pix/features.py'>

In [16]:
from pcm_pix.features import load_mesh_tables, make_nn_dataset

mesh_tables = load_mesh_tables(CFG, base_dir="data")
ds = make_nn_dataset(mesh_tables, wl=CFG["wl"])

logger.info("dataset: df=%s, data_0=%s, data_1=%s", len(ds.df), len(ds.data_0), len(ds.data_1))
logger.info("X_0=%s y_0=%s | X_1=%s y_1=%s", ds.X_0.shape, ds.y_0.shape, ds.X_1.shape, ds.y_1.shape)

2026-03-03 12:24:20,159 | INFO | dataset: df=98390, data_0=60360, data_1=38030
2026-03-03 12:24:20,160 | INFO | X_0=(60360, 3) y_0=(60360, 4) | X_1=(38030, 3) y_1=(38030, 4)


In [ ]:
from pcm_pix.nn_surrogate import train_or_load_surrogates, get_device

CFG["device"] = get_device()
CFG["epochs"] = 300000   # для первой проверки можно 200-500, чтобы быстро
CFG["lr"] = 1e-3

sur0, sur1 = train_or_load_surrogates(ds, run, CFG, force_train=False)
logger.info("surrogates ready")

2026-03-03 14:00:16,852 | INFO | loading surrogates from outputs/PCM_bagel_2025/models
2026-03-03 14:00:16,872 | INFO | surrogates ready


In [33]:
import numpy as np

def wrap_to_pi(x):
    # приводит разность фаз к диапазону [-pi, pi]
    return (x + np.pi) % (2 * np.pi) - np.pi

def evaluate_surrogate(ds_df, sur, n=5000, seed=42, label="am"):
    """
    ds_df: ds.data_0 или ds.data_1 (из make_nn_dataset)
    sur: sur0 или sur1 (из train_or_load_surrogates или load_legacy_surrogates)
    """
    rng = np.random.default_rng(seed)
    if n is None or n >= len(ds_df):
        idx = np.arange(len(ds_df))
    else:
        idx = rng.choice(len(ds_df), size=n, replace=False)

    df = ds_df.iloc[idx]

    # true
    R_true = df["R"].to_numpy()
    T_true = df["T1"].to_numpy()
    phiR_true = df["phi_R"].to_numpy()
    phiT_true = df["phi_T1"].to_numpy()

    # predict in (cos,sin)
    pred = sur.predict(df["a"].to_list(), df["d"].to_list(), df["b"].to_list())
    Rcos_p, Rsin_p, Tcos_p, Tsin_p = pred[:, 0], pred[:, 1], pred[:, 2], pred[:, 3]

    R_pred = np.sqrt(Rcos_p**2 + Rsin_p**2)
    T_pred = np.sqrt(Tcos_p**2 + Tsin_p**2)
    phiR_pred = np.arctan2(Rsin_p, Rcos_p)
    phiT_pred = np.arctan2(Tsin_p, Tcos_p)

    # ошибки (амплитуды)
    R_mae = float(np.mean(np.abs(R_pred - R_true)))
    R_rmse = float(np.sqrt(np.mean((R_pred - R_true) ** 2)))
    T_mae = float(np.mean(np.abs(T_pred - T_true)))
    T_rmse = float(np.sqrt(np.mean((T_pred - T_true) ** 2)))

    # ошибки (фазы) с учётом периодичности
    dphiR = wrap_to_pi(phiR_pred - phiR_true)
    dphiT = wrap_to_pi(phiT_pred - phiT_true)
    phiR_mae = float(np.mean(np.abs(dphiR)))
    phiR_rmse = float(np.sqrt(np.mean(dphiR ** 2)))
    phiT_mae = float(np.mean(np.abs(dphiT)))
    phiT_rmse = float(np.sqrt(np.mean(dphiT ** 2)))

    # “физические” проверки диапазонов
    R_out_of_01 = int(np.sum((R_pred < 0) | (R_pred > 1)))
    T_out_of_01 = int(np.sum((T_pred < 0) | (T_pred > 1)))

    print(f"[{label}] n={len(df)}")
    print(f"[{label}] R:  MAE={R_mae:.6f} RMSE={R_rmse:.6f} | out_of_[0,1]={R_out_of_01}")
    print(f"[{label}] T1: MAE={T_mae:.6f} RMSE={T_rmse:.6f} | out_of_[0,1]={T_out_of_01}")
    print(f"[{label}] phi_R:  MAE={phiR_mae:.6f} rad  RMSE={phiR_rmse:.6f} rad")
    print(f"[{label}] phi_T1: MAE={phiT_mae:.6f} rad  RMSE={phiT_rmse:.6f} rad")

    # вернуть числа (удобно потом логировать/сохранять)
    return {
        "n": int(len(df)),
        "R_mae": R_mae, "R_rmse": R_rmse,
        "T_mae": T_mae, "T_rmse": T_rmse,
        "phiR_mae": phiR_mae, "phiR_rmse": phiR_rmse,
        "phiT_mae": phiT_mae, "phiT_rmse": phiT_rmse,
        "R_out_of_01": R_out_of_01,
        "T_out_of_01": T_out_of_01,
    }

# пример использования:
m_am = evaluate_surrogate(ds.data_0, sur0, n=5000, seed=42, label="am")
m_cr = evaluate_surrogate(ds.data_1, sur1, n=5000, seed=42, label="cr")

[am] n=5000
[am] R:  MAE=0.002524 RMSE=0.005561 | out_of_[0,1]=14
[am] T1: MAE=0.003090 RMSE=0.007655 | out_of_[0,1]=4
[am] phi_R:  MAE=0.024273 rad  RMSE=0.094007 rad
[am] phi_T1: MAE=0.014583 rad  RMSE=0.115491 rad
[cr] n=5000
[cr] R:  MAE=0.002616 RMSE=0.005520 | out_of_[0,1]=17
[cr] T1: MAE=0.002998 RMSE=0.007327 | out_of_[0,1]=1
[cr] phi_R:  MAE=0.023531 rad  RMSE=0.097879 rad
[cr] phi_T1: MAE=0.016188 rad  RMSE=0.123701 rad


In [ ]:
import numpy as np

PRESETS = {
    "pos_2026_03_03_example": np.array([999.9999832237819, 984.3026291343397, 958.2542604775822, 936.6180434251005, 
                                        993.4814923872052, 999.9999543918377, 950.8023314568704, 999.7122801728465, 
                                        812.6419505503554, 900.387276453811, 954.4390294470007, 650.2800698156437, 
                                        662.5287955357173, 681.2858693125721, 700.083406735213, 726.3045463730607, 
                                        931.6468661009544, 770.9736892629372, 819.773820550387, 569.5143584777072, 
                                        520.9010616227479, 480.96992943410584, 314.16505765104455, 311.1638267139905, 
                                        310.6361359995624, 323.64108002976633, 469.1378222786072, 530.8057496399709, 
                                        519.5742333256374, 690.5658277235785, 50.05271853544764, 7.789435407260271, 
                                        37.236000971268254, 5.051730929403151, 3.8741672878189197, 0.9047503198059405, 
                                        0.9887957678343591]),
}
from pcm_pix.optimize_pso import save_solution

for name, pos in PRESETS.items():
    save_solution(run, name=name, pos=pos, cost=None, meta={"source": "old_notebook"})

2026-03-03 13:02:46,736 | INFO | saved solution pos_2026_03_03_example.json


In [ ]:
import numpy as np
import pandas as pd
from pcm_pix.optimize_pso import run_pso_until

def pso_cost_once(sur0, sur1, base_cfg, run, c1, c2, w, seed):
    cfg = dict(base_cfg)

    # быстрый режим подбора (на 1 час)
    cfg["pso_threshold"] = 999999
    cfg["pso_max_restarts"] = 0
    cfg["pso_n_particles"] = 250
    cfg["pso_iters"] = 60

    cfg["pso_c1"] = float(c1)
    cfg["pso_c2"] = float(c2)
    cfg["pso_w"]  = float(w)

    np.random.seed(seed)
    best = run_pso_until(sur0, sur1, cfg, run)
    return float(best.cost)

In [36]:
rng = np.random.default_rng(42)

n_trials = 80        # ~40 минут (если 1 прогон ~15-20 сек)
repeats = 2          # чтобы не “повезло” случайно

rows = []
for t in range(n_trials):
    # диапазоны адекватные для PSO
    c1 = rng.uniform(0.1, 1.5)
    c2 = rng.uniform(0.1, 1.5)
    w  = rng.uniform(0.3, 0.95)

    costs = []
    for r in range(repeats):
        seed = 1000*t + r
        costs.append(pso_cost_once(sur0, sur1, CFG, run, c1, c2, w, seed))

    rows.append({
        "c1": c1, "c2": c2, "w": w,
        "cost_median": float(np.median(costs)),
        "cost_mean": float(np.mean(costs)),
        "cost_min": float(np.min(costs)),
    })

    if (t + 1) % 10 == 0:
        logger.info("random search %s/%s done", t + 1, n_trials)

df_rs = pd.DataFrame(rows).sort_values("cost_median").reset_index(drop=True)
df_rs.head(10)
df_rs.to_csv(run.results / "pso_random_search.csv", index=False)

2026-03-03 14:07:13,465 | INFO | PSO start: particles=300 iters=80 dims=37
2026-03-03 14:07:13,466 - pyswarms.single.global_best - INFO - Optimize for 80 iters with {'c1': 0.3, 'c2': 0.3, 'w': 0.6}
pyswarms.single.global_best: 100%|██████████|80/80, best_cost=6.55  
2026-03-03 14:07:14,877 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 6.551106084458357, best pos: [9.53804149e+02 6.68381756e+02 7.65364167e+02 6.85096229e+02
 9.10851417e+02 7.97323722e+02 8.12608300e+02 6.42893416e+02
 5.30766660e+02 7.88784110e+02 8.43167627e+02 4.48388432e+02
 4.13291403e+02 5.86788514e+02 6.05778469e+02 6.47344814e+02
 3.20074381e+02 5.33381910e+02 5.12315396e+02 3.19261781e+02
 5.21837106e+02 5.80297541e+02 2.06257036e+02 3.09747739e+02
 1.86606146e+02 2.34810528e+02 2.65162062e+02 1.42436208e+02
 3.67352689e+02 3.01532066e+02 7.84315282e+01 3.23084821e+02
 4.04312185e+02 5.30147854e+00 4.92414948e+00 8.88766198e-01
 9.24714754e-01]
2026-03-03 14:07:14,881 | INFO | PSO don

,c1,c2,w,cost_mean,cost_median,cost_min
0,0.5,0.7,0.6,4.746646,4.815338,4.471282
1,0.7,0.7,0.6,5.165783,5.025265,4.913126
2,0.3,0.7,0.6,5.136093,5.050189,4.737356
3,0.7,0.3,0.8,5.876454,5.586024,5.085815
4,0.7,0.5,0.6,5.567571,5.708497,5.283123
5,0.5,0.5,0.8,6.054534,5.712682,5.294312
6,0.5,0.3,0.9,5.860661,5.888074,5.647294
7,0.5,0.3,0.8,5.859772,5.908485,5.133553
8,0.3,0.5,0.8,6.063002,5.941732,4.897335
9,0.7,0.5,0.8,6.091184,6.058892,5.848538


In [ ]:
top = df_rs.head(3)

def refine_grid_around(row, dc1=0.2, dc2=0.2, dw=0.1):
    c1, c2, w = float(row.c1), float(row.c2), float(row.w)
    grid_c1 = [c1-dc1, c1-dc1/2, c1, c1+dc1/2, c1+dc1]
    grid_c2 = [c2-dc2, c2-dc2/2, c2, c2+dc2/2, c2+dc2]
    grid_w  = [w-dw,  w-dw/2,  w,  w+dw/2,  w+dw]
    grid_c1 = [float(np.clip(x, 0.05, 2.5)) for x in grid_c1]
    grid_c2 = [float(np.clip(x, 0.05, 2.5)) for x in grid_c2]
    grid_w  = [float(np.clip(x, 0.10, 0.99)) for x in grid_w]
    return grid_c1, grid_c2, grid_w

from itertools import product

rows2 = []
repeats2 = 3

for i in range(len(top)):
    grid_c1, grid_c2, grid_w = refine_grid_around(top.iloc[i])

    for c1, c2, w in product(grid_c1, grid_c2, grid_w):
        costs = []
        for r in range(repeats2):
            seed = 900000 + 1000*i + r
            costs.append(pso_cost_once(sur0, sur1, CFG, run, c1, c2, w, seed))

        rows2.append({
            "base_rank": i,
            "c1": c1, "c2": c2, "w": w,
            "cost_median": float(np.median(costs)),
            "cost_mean": float(np.mean(costs)),
            "cost_min": float(np.min(costs)),
        })

df_ref = pd.DataFrame(rows2).sort_values("cost_median").reset_index(drop=True)
df_ref.head(10)
df_ref.to_csv(run.results / "pso_refine.csv", index=False)

2026-03-03 14:10:42,453 | INFO | saved outputs/PCM_bagel_2025/results/pso_hyperopt.csv


In [ ]:
best = df_ref.iloc[0]
CFG["pso_c1"] = float(best.c1)
CFG["pso_c2"] = float(best.c2)
CFG["pso_w"]  = float(best.w)

logger.info("FINAL PSO hyperparams: c1=%s c2=%s w=%s", CFG["pso_c1"], CFG["pso_c2"], CFG["pso_w"])

# боевые
CFG["pso_threshold"] = 4
CFG["pso_max_restarts"] = 20
CFG["pso_n_particles"] = 3000
CFG["pso_iters"] = 1000

best_pso = run_pso_until(sur0, sur1, CFG, run)
best_pso

In [ ]:
from pcm_pix.optimize_pso import run_pso_until, run_de_full

# 1) PSO как в ноутбуке (порог + рестарты)
CFG["pso_threshold"] = 4
CFG["pso_max_restarts"] = 20
CFG["pso_n_particles"] = 3000
CFG["pso_iters"] = 1000
CFG["pso_c1"] = 0.5
CFG["pso_c2"] = 0.3
CFG["pso_w"] = 0.9

best_pso = run_pso_until(sur0, sur1, CFG, run)

In [ ]:
# 2) DE как в ноутбуке (init_ar + vectorized + callback)
CFG["de_init_mode"] = "init_ar"
CFG["de_init_N"] = 1000
CFG["de_mutation"] = 0.99
CFG["de_recombination"] = 0.1
CFG["de_maxiter"] = 1000000
CFG["de_popsize"] = 5000
CFG["de_tol"] = 1e-12
CFG["de_atol"] = 1e-12
CFG["de_polish"] = True
CFG["de_callback_every"] = 1000
CFG["de_use_linear_constraint"] = True  # если хочешь именно constraints-ветку — поставь True

best = run_de_full(sur0, sur1, CFG, run, pos=best_pso.pos)
best

In [3]:
from pcm_pix.run import start_run

run = start_run(outputs_dir="outputs", run_name=CFG["run_name"])
logger = run.logger
logger.info("main.ipynb started")

2026-03-03 12:07:56,122 | INFO | run_dir=/home/slava/КВАНТЭЛ/GIT/PCM_pix/outputs/PCM_bagel_2025
2026-03-03 12:07:56,127 | INFO | main.ipynb started


In [4]:
# TODO: data = load_inputs(CFG)

In [5]:
# TODO: sur0, sur1 = train_or_load(...)

In [6]:
# TODO: best = optimize(...)
# TODO: export_gds(...)